# BetHero — full-budget training on Colab's free GPU

Trains all 8 model_keys (BTTS, Cards_3.5, Goals_1.5/2.5/3.5, Corners_7.5/9.5, 1x2) at `n_trials=20` using the local feature-cache snapshot — no live Supabase queries during training.

**One-time setup before running this notebook:**
1. Push the repo to a GitHub repo (private is fine) — Colab clones from there.
2. Upload `data_snapshot/` (131MB) to a folder in your Google Drive, e.g. `MyDrive/bethero/data_snapshot/`.
3. Add two Colab secrets (key icon in the left sidebar): `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`. Never hardcode these in a cell.
4. Runtime → Change runtime type → **T4 GPU** (or whatever free GPU is offered).

Trained models are written straight to a Drive-mounted folder as each model_key finishes — a disconnect only loses the one market in flight, everything already done is safe. Re-running the notebook skips model_keys that already have a saved `StackingEnsemble_*.joblib` in that folder.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Replace with your repo URL. For a private repo, use a token:
#   https://<token>@github.com/<you>/<repo>.git
REPO_URL = "https://github.com/<you>/<repo>.git"

!git clone $REPO_URL /content/bethero
%cd /content/bethero

In [ ]:
# Install everything except torch — Colab already ships a CUDA-enabled torch,
# and requirements.txt pins the CPU-only wheel (torch==2.2.2+cpu), which
# would silently DISABLE the GPU if installed over Colab's own build.
!grep -v '^torch==' requirements.txt | grep -v 'download.pytorch.org' > /tmp/reqs_no_torch.txt
!pip install -q -r /tmp/reqs_no_torch.txt

In [ ]:
import os
from google.colab import userdata

os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = userdata.get("SUPABASE_SERVICE_ROLE_KEY")

# Trained models land directly on Drive — survives a disconnect mid-run.
DRIVE_MODEL_DIR = "/content/drive/MyDrive/bethero/models"
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
os.environ["MODEL_DIR"] = DRIVE_MODEL_DIR

DATA_DIR = "/content/drive/MyDrive/bethero/data_snapshot"
assert os.path.isdir(DATA_DIR), f"Upload data_snapshot/ to {DATA_DIR} first"
print("MODEL_DIR ->", DRIVE_MODEL_DIR)
print("DATA_DIR  ->", DATA_DIR)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — check Runtime > Change runtime type")

In [ ]:
import sys
sys.path.insert(0, "/content/bethero")

from pathlib import Path
from app.ml.models import MODEL_DIR
from app.ml.training_pipeline_logic import train_market

MODEL_KEYS = [
    "BTTS", "Cards_3.5",
    "Goals_1.5", "Goals_2.5", "Goals_3.5",
    "Corners_7.5", "Corners_9.5",
    "1x2",
]
N_TRIALS = 20
N_META_TRIALS = 20

print("MODEL_DIR resolved to:", MODEL_DIR)
assert str(MODEL_DIR) == os.environ["MODEL_DIR"], "MODEL_DIR env var must be set BEFORE importing app.ml.* — restart runtime and re-run from the top"

In [ ]:
import time

for model_key in MODEL_KEYS:
    ensemble_path = MODEL_DIR / f"StackingEnsemble_football_{model_key}.joblib"
    if ensemble_path.exists():
        print(f"=== {model_key}: already trained, skipping ({ensemble_path}) ===")
        continue

    print(f"\n=== {model_key} — starting ===", flush=True)
    t0 = time.monotonic()
    results = train_market(
        "football", model_key,
        n_trials=N_TRIALS, n_meta_trials=N_META_TRIALS,
        data_dir=Path(DATA_DIR),
    )
    elapsed = time.monotonic() - t0
    for r in results:
        print(f"  {r.model_name}: {r.status} {r.error or ''} ({r.duration_s}s)")
        if r.calibration_table is not None:
            print(r.calibration_table.to_string(index=False))
    print(f"{model_key} done in {elapsed/60:.1f} min", flush=True)

print("\n=== ALL MARKETS DONE ===")

## After this finishes
Trained model files are already sitting in your Drive at `MyDrive/bethero/models/` — copy that folder over your local `models/` directory (or point `MODEL_DIR` at it directly) to make them available to `Predictor`.